In [29]:
import numpy as np
import pandas as pd

from stanbkt.models.simple import StandardBKT

In [30]:


def sim_simple_BKT(
    rng=np.random.default_rng(0),
    nStudents: int = 10,
    nProblems: int = 20,
    nKcs: int = 1,
    prior=0.1,
    learn=0.01,
    forget=0.05,
    guess=0.2,
    slip=0.1,
    kc_sequence=None,
):

    def _param_to_vec(x, name):
        arr = np.asarray(x, dtype=float)
        if arr.size == 1:
            arr = np.repeat(arr, nKcs)
        if arr.size != nKcs:
            raise ValueError(f"{name} must be scalar or length nKcs")
        return arr

    prior_vec = _param_to_vec(prior, "prior")
    learn_vec = _param_to_vec(learn, "learn")
    forget_vec = _param_to_vec(forget, "forget")
    guess_vec = _param_to_vec(guess, "guess")
    slip_vec = _param_to_vec(slip, "slip")

    if kc_sequence is None:
        kc_sequence = rng.integers(0, nKcs, size=nProblems)
    else:
        kc_sequence = np.asarray(kc_sequence, dtype=int)
        if kc_sequence.shape[0] != nProblems:
            raise ValueError("kc_sequence must have length nProblems")
        if kc_sequence.min() < 0 or kc_sequence.max() >= nKcs:
            raise ValueError("kc_sequence entries must be in [0, nKcs-1]")

    knowledge = rng.random(size=(nStudents, nKcs)) < prior_vec
    correctness = np.zeros((nStudents, nProblems), dtype=int)
    states = np.zeros((nStudents, nProblems), dtype=int)

    for t in range(nProblems):
        kc = kc_sequence[t]
        for s in range(nStudents):
            knows_before = knowledge[s, kc]
            if knows_before:
                correct = int(rng.random() >= slip_vec[kc])
            else:
                correct = int(rng.random() < guess_vec[kc])

            correctness[s, t] = correct

            if knows_before:
                knowledge[s, kc] = rng.random() >= forget_vec[kc]
            else:
                knowledge[s, kc] = rng.random() < learn_vec[kc]

            states[s, t] = knowledge[s, kc]

    return (
        correctness,
        states,
        kc_sequence,
        nStudents,
        nProblems,
        nKcs,
    )

In [31]:
# 1) Generate synthetic data (from scratch.ipynb logic)
(
    correctness,
    states,
    kc_sequence,
    n_students,
    n_problems,
    n_kcs,
 ) = sim_simple_BKT(
    rng=np.random.default_rng(123),
    nStudents=50,
    nProblems=30,
    nKcs=3,
    prior=0.4,
    learn=0.04,
    forget=0.01,
    guess=0.2,
    slip=0.1,
    kc_sequence=None,
)

correctness.shape, n_kcs

((50, 30), 3)

In [32]:
# 2) Build long-format dataframe expected by StanBKT
student_idx, problem_idx = np.indices(correctness.shape)

data_df = pd.DataFrame(
    {
        "student_id": student_idx.ravel().astype(str),
        "problem_id": problem_idx.ravel().astype(str),
        "correct": correctness.ravel().astype(int),
        "kc_id": kc_sequence[problem_idx.ravel()].astype(str),
    }
)

data_df.head()

,student_id,problem_id,correct,kc_id
0,0,0,0,0
1,0,1,0,2
2,0,2,1,1
3,0,3,0,0
4,0,4,1,2


In [33]:
data_df['kc_id'].value_counts()

kc_id
0    600
2    550
1    350
Name: count, dtype: int64

In [34]:
# 3) Instantiate and fit the StandardBKT model
model = StandardBKT()
model.fit(data_df)

11:33:46 - cmdstanpy - INFO - CmdStan start processing
chain 1:   0%|          | 0/3500 [00:00<?, ?it/s, (Warmup)]





chain 1:   3%|▎         | 100/3500 [00:00<00:16, 205.49it/s, (Warmup)]





chain 1:   6%|▌         | 200/3500 [00:00<00:13, 242.73it/s, (Warmup)]





chain 1:   9%|▊         | 300/3500 [00:01<00:10, 305.24it/s, (Warmup)]


chain 1:  14%|█▍        | 500/3500 [00:01<00:05, 519.84it/s, (Warmup)]


chain 1:  23%|██▎       | 800/3500 [00:01<00:03, 793.06it/s, (Warmup)]


chain 1:  26%|██▌       | 900/3500 [00:01<00:03, 836.05it/s, (Warmup)]

chain 1:  29%|██▊       | 1000/3500 [00:01<00:03, 826.53it/s, (Warmup)]


chain 1:  34%|███▍      | 1200/3500 [00:02<00:02, 825.88it/s, (Warmup)]

chain 1:  37%|███▋      | 1300/3500 [00:02<00:02, 828.50it/s, (Warmup)]





chain 1:  43%|████▎     | 1500/3500 [00:02<00:02, 762.20it/s, (Warmup)]


chain 1:  49%|████▊     | 1700/3500 [00:02<00:02, 770.79it/s, (Warmup)]


chain 1:  51%|█████▏    | 1800/3500 [00:02<00:02, 669.33it/s, (Wa


11:33:52 - cmdstanpy - INFO - CmdStan done processing.
11:33:52 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/3500 [00:00<?, ?it/s, (Warmup)]




chain 1:   3%|▎         | 100/3500 [00:00<00:10, 338.28it/s, (Warmup)]


chain 1:   6%|▌         | 200/3500 [00:00<00:07, 416.50it/s, (Warmup)]


chain 1:   9%|▊         | 300/3500 [00:00<00:07, 413.35it/s, (Warmup)]


chain 1:  11%|█▏        | 400/3500 [00:00<00:06, 474.29it/s, (Warmup)]


chain 1:  14%|█▍        | 500/3500 [00:01<00:05, 576.55it/s, (Warmup)]

chain 1:  20%|██        | 700/3500 [00:01<00:04, 689.61it/s, (Warmup)]


chain 1:  23%|██▎       | 800/3500 [00:01<00:03, 764.90it/s, (Warmup)]


chain 1:  26%|██▌       | 900/3500 [00:01<00:03, 715.50it/s, (Warmup)]


chain 1:  29%|██▊       | 1000/3500 [00:01<00:04, 599.14it/s, (Warmup)]


chain 1:  31%|███▏      | 1100/3500 [00:01<00:03, 651.13it/s, (Warmup)]


chain 1:  37%|███▋      | 1300/3500 [00:02<00:02, 822.24it/s, (Warmup)]




chain 1:  43%|████▎     | 1500/3500 [00:02<00:02, 948.94it/s, (Warmup)]


chain 1:  49%|████▊     | 1700/3500 [00:02<00:01, 1036.


11:33:57 - cmdstanpy - INFO - CmdStan done processing.
11:33:57 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/3500 [00:00<?, ?it/s, (Warmup)]



chain 1:   3%|▎         | 100/3500 [00:00<00:10, 338.20it/s, (Warmup)]


chain 1:   6%|▌         | 200/3500 [00:00<00:06, 506.81it/s, (Warmup)]

chain 1:   9%|▊         | 300/3500 [00:00<00:05, 603.95it/s, (Warmup)]


chain 1:  11%|█▏        | 400/3500 [00:00<00:04, 646.88it/s, (Warmup)]


chain 1:  14%|█▍        | 500/3500 [00:00<00:04, 660.01it/s, (Warmup)]


chain 1:  20%|██        | 700/3500 [00:01<00:03, 717.19it/s, (Warmup)]


chain 1:  23%|██▎       | 800/3500 [00:01<00:03, 747.02it/s, (Warmup)]


chain 1:  26%|██▌       | 900/3500 [00:01<00:03, 691.22it/s, (Warmup)]


chain 1:  31%|███▏      | 1100/3500 [00:01<00:03, 723.33it/s, (Warmup)]




chain 1:  34%|███▍      | 1200/3500 [00:01<00:03, 727.87it/s, (Warmup)]


chain 1:  37%|███▋      | 1300/3500 [00:01<00:02, 736.84it/s, (Warmup)]


chain 1:  40%|████      | 1400/3500 [00:02<00:02, 759.86it/s, (Warmup)]



chain 1:  46%|████▌     | 1600/3500 [00:02<00:02, 752.2


11:34:02 - cmdstanpy - INFO - CmdStan done processing.


In [35]:
# 4) Predict hidden states
hidden_state_df = model.predict(data_df)
hidden_state_df

11:34:02 - cmdstanpy - INFO - Chain [1] start processing
11:34:02 - cmdstanpy - INFO - Chain [2] start processing
11:34:02 - cmdstanpy - INFO - Chain [3] start processing
11:34:02 - cmdstanpy - INFO - Chain [4] start processing
11:34:03 - cmdstanpy - INFO - Chain [4] done processing
11:34:03 - cmdstanpy - INFO - Chain [3] done processing
11:34:03 - cmdstanpy - INFO - Chain [2] done processing
11:34:03 - cmdstanpy - INFO - Chain [1] done processing
11:34:03 - cmdstanpy - WARNING - Sample doesn't contain draws from warmup iterations, rerun sampler with "save_warmup=True".
11:34:03 - cmdstanpy - INFO - Chain [1] start processing
11:34:03 - cmdstanpy - INFO - Chain [2] start processing
11:34:03 - cmdstanpy - INFO - Chain [3] start processing
11:34:03 - cmdstanpy - INFO - Chain [4] start processing
11:34:04 - cmdstanpy - INFO - Chain [3] done processing
11:34:04 - cmdstanpy - INFO - Chain [2] done processing
11:34:04 - cmdstanpy - INFO - Chain [1] done processing
11:34:04 - cmdstanpy - INFO

,variable,mean,std,median,2.5%,97.5%,kc_id
0,"pKnow[1,1]",0.525289,0.077541,0.527645,0.372113,0.674997,0
1,"pCorrectness[1,1]",0.543541,0.054127,0.543716,0.435433,0.647261,0
2,"pKnow[1,2]",0.145790,0.046478,0.141303,0.072718,0.252847,0
3,"pCorrectness[1,2]",0.276941,0.035450,0.275181,0.211371,0.350209,0
4,"pKnow[1,3]",0.029013,0.013508,0.026696,0.009955,0.062232,0
...,...,...,...,...,...,...,...
2995,"pCorrectness[50,5]",0.231637,0.038487,0.228289,0.162434,0.314082,1
2996,"pKnow[50,6]",0.015392,0.017706,0.009609,0.000643,0.060812,1
2997,"pCorrectness[50,6]",0.214515,0.042134,0.212272,0.136628,0.302180,1
2998,"pKnow[50,7]",0.010189,0.014853,0.004638,0.000153,0.051362,1


In [36]:
# 5) Predict smoothed hidden states
smoothed_state_df = model.predict_smoothed_states(data_df)
smoothed_state_df

11:34:05 - cmdstanpy - INFO - Chain [1] start processing
11:34:05 - cmdstanpy - INFO - Chain [2] start processing
11:34:05 - cmdstanpy - INFO - Chain [3] start processing
11:34:05 - cmdstanpy - INFO - Chain [4] start processing
11:34:06 - cmdstanpy - INFO - Chain [3] done processing
11:34:06 - cmdstanpy - INFO - Chain [4] done processing
11:34:06 - cmdstanpy - INFO - Chain [1] done processing
11:34:06 - cmdstanpy - INFO - Chain [2] done processing
11:34:06 - cmdstanpy - WARNING - Sample doesn't contain draws from warmup iterations, rerun sampler with "save_warmup=True".
11:34:06 - cmdstanpy - INFO - Chain [1] start processing
11:34:06 - cmdstanpy - INFO - Chain [2] start processing
11:34:06 - cmdstanpy - INFO - Chain [3] start processing
11:34:06 - cmdstanpy - INFO - Chain [4] start processing
11:34:06 - cmdstanpy - INFO - Chain [2] done processing
11:34:06 - cmdstanpy - INFO - Chain [1] done processing
11:34:06 - cmdstanpy - INFO - Chain [4] done processing
11:34:06 - cmdstanpy - INFO

,variable,mean,std,median,2.5%,97.5%,kc_id
0,"know_probs[1,1]",0.009950,0.006642,0.008556,0.001478,0.026687,0
1,"know_probs[1,2]",0.005131,0.003698,0.004250,0.000686,0.014689,0
2,"know_probs[1,3]",0.005087,0.003758,0.004197,0.000651,0.015005,0
3,"know_probs[1,4]",0.000785,0.000651,0.000602,0.000082,0.002523,0
4,"know_probs[1,5]",0.000141,0.000138,0.000098,0.000011,0.000514,0
...,...,...,...,...,...,...,...
1495,"know_probs[50,3]",0.004938,0.008211,0.002453,0.000037,0.024761,1
1496,"know_probs[50,4]",0.001063,0.001662,0.000542,0.000018,0.005498,1
1497,"know_probs[50,5]",0.000493,0.000833,0.000216,0.000008,0.002727,1
1498,"know_probs[50,6]",0.000568,0.000920,0.000263,0.000008,0.002914,1


In [51]:
correctness[0]

array([0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1,
       0, 0, 0, 1, 1, 0, 1, 0])

In [44]:
hidden_state_df.loc[hidden_state_df['variable'].str.startswith('pKnow[1,'),'mean'].reset_index(drop=True)

0     0.525289
1     0.145790
2     0.029013
3     0.128348
4     0.025949
5     0.007587
6     0.004560
7     0.004054
8     0.003968
9     0.003952
10    0.003949
11    0.003949
12    0.570393
13    0.210962
14    0.523402
15    0.809726
16    0.938776
17    0.737385
18    0.908555
19    0.969587
20    0.987662
21    0.992735
22    0.956362
23    0.538887
24    0.812045
25    0.926728
26    0.961134
27    0.970531
28    0.973080
29    0.973784
Name: mean, dtype: float64

In [46]:
smoothed_state_df.loc[smoothed_state_df["variable"].str.startswith("know_probs[1,"), "mean"]

0       0.009950
1       0.005131
2       0.005087
3       0.000785
4       0.000141
5       0.000041
6       0.000024
7       0.000022
8       0.000023
9       0.000033
10      0.000099
11      0.000559
600     0.862694
601     0.957017
602     0.979210
603     0.984352
604     0.984673
605     0.993372
606     0.995463
607     0.995616
608     0.994139
609     0.987611
610     0.987750
1150    0.995981
1151    0.997493
1152    0.997470
1153    0.995984
1154    0.989719
1155    0.963798
1156    0.852494
Name: mean, dtype: float64